# Restaurants 외 Food 업체 7,203개 범위 확장 검증

## 검증 목적

Restaurants에는 포함되지 않고 Food에만 포함된 12,309개 업체가
미식 콘텐츠 분석 범위에 적합한지 확인한다.

다음 항목을 검증한다.

기존 Restaurants 범위에 Food 카테고리를 추가할 경우 다음 항목을 확인한다.

1. 추가되는 Food-only 업체 수
2. Food-only 업체의 세부 카테고리 구성
3. 추가되는 리뷰와 사용자 수
4. 파워 리뷰어 코호트 증가량
5. 이탈 라벨 변화
6. 분석 범위 확장 적용 여부

현재 검증 범위는 다음과 같다.

- 기존 범위: Restaurants
- 확장 후보: Restaurants 또는 Food
- 최종 확장 여부는 분석 결과를 보고 결정한다.

In [11]:
from pathlib import Path
from collections import Counter
import gc

import numpy as np
import pandas as pd


# 현재 실행 위치
current_path = Path.cwd().resolve()

# data/interim 폴더를 기준으로 프로젝트 루트 탐색
PROJECT_ROOT = next(
    (
        path
        for path in [
            current_path,
            *current_path.parents
        ]
        if (
            path
            / "data"
            / "interim"
        ).exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾을 수 없습니다. "
        "상위 경로에 data/interim 폴더가 있는지 확인하세요."
    )

print("현재 실행 위치:", current_path)
print("프로젝트 루트:", PROJECT_ROOT)

현재 실행 위치: C:\Users\playdata2\SKN34-2nd-5Team\notebooks
프로젝트 루트: C:\Users\playdata2\SKN34-2nd-5Team


In [12]:
RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

REPORT_TABLE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

VALIDATION_CACHE_DIR = (
    INTERIM_DIR
    / "validation"
)

BUSINESS_JSON_PATH = (
    RAW_DIR
    / "yelp_academic_dataset_business.json"
)

REVIEW_JSON_PATH = (
    RAW_DIR
    / "yelp_academic_dataset_review.json"
)

RESTAURANT_REVIEW_PATH = (
    INTERIM_DIR
    / "restaurant_reviews.parquet"
)

FOOD_ONLY_MONTHLY_CACHE_PATH = (
    VALIDATION_CACHE_DIR
    / "food_only_monthly_activity_candidate_v01.parquet"
)

FOOD_ONLY_BUSINESS_REVIEW_CACHE_PATH = (
    VALIDATION_CACHE_DIR
    / "food_only_business_review_counts_candidate_v01.parquet"
)

RESTAURANT_MONTHLY_CACHE_PATH = (
    VALIDATION_CACHE_DIR
    / "restaurant_monthly_activity_validation_v01.parquet"
)

REPORT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VALIDATION_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Business JSON:", BUSINESS_JSON_PATH.exists())
print("Review JSON:", REVIEW_JSON_PATH.exists())
print("Restaurant 리뷰:", RESTAURANT_REVIEW_PATH.exists())

assert BUSINESS_JSON_PATH.exists()
assert REVIEW_JSON_PATH.exists()
assert RESTAURANT_REVIEW_PATH.exists()

print("필수 파일 경로 검증 통과")

Business JSON: True
Review JSON: True
Restaurant 리뷰: True
필수 파일 경로 검증 통과


In [13]:
# 3. Business 데이터 불러오기
business_df = pd.read_json(
    BUSINESS_JSON_PATH,
    lines=True
)

business_df = (
    business_df[
        [
            "business_id",
            "name",
            "city",
            "state",
            "categories"
        ]
    ]
    .copy()
)

print("Business 데이터 크기:", business_df.shape)
print("고유 업체:", business_df["business_id"].nunique())
print("업체 ID 중복:", business_df["business_id"].duplicated().sum())
print("카테고리 결측:", business_df["categories"].isna().sum())

assert len(business_df) == 150_346
assert business_df["business_id"].is_unique

business_df.head()

Business 데이터 크기: (150346, 5)
고유 업체: 150346
업체 ID 중복: 0
카테고리 결측: 103


,business_id,name,city,state,categories
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ",Santa Barbara,CA,"Doctors, Traditional Chinese Medicine, Naturop..."
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,Affton,MO,"Shipping Centers, Local Services, Notaries, Ma..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,Tucson,AZ,"Department Stores, Shopping, Fashion, Home & G..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,Green Lane,PA,"Brewpubs, Breweries, Food"


In [14]:
# 4. Restaurants·Food 범위 생성
business_df["category_list"] = (
    business_df["categories"]
    .fillna("")
    .str.split(", ")
)

business_df["is_restaurant"] = (
    business_df["category_list"]
    .apply(
        lambda categories:
            "Restaurants" in categories
    )
)

business_df["is_food"] = (
    business_df["category_list"]
    .apply(
        lambda categories:
            "Food" in categories
    )
)

business_df["is_restaurant_and_food"] = (
    business_df["is_restaurant"]
    & business_df["is_food"]
)

business_df["is_food_only"] = (
    business_df["is_food"]
    & ~business_df["is_restaurant"]
)

business_df["is_expanded_food"] = (
    business_df["is_restaurant"]
    | business_df["is_food"]
)

In [15]:
# 5. 업체 범위 집계
scope_business_summary_df = pd.DataFrame(
    {
        "scope": [
            "전체 업체",
            "Restaurants",
            "Food",
            "Restaurants와 Food 교집합",
            "Food Only",
            "Restaurants 또는 Food"
        ],
        "business_count": [
            len(business_df),
            int(
                business_df[
                    "is_restaurant"
                ].sum()
            ),
            int(
                business_df[
                    "is_food"
                ].sum()
            ),
            int(
                business_df[
                    "is_restaurant_and_food"
                ].sum()
            ),
            int(
                business_df[
                    "is_food_only"
                ].sum()
            ),
            int(
                business_df[
                    "is_expanded_food"
                ].sum()
            )
        ]
    }
)

scope_business_summary_df[
    "business_rate_pct"
] = (
    scope_business_summary_df[
        "business_count"
    ]
    / len(business_df)
    * 100
).round(2)

scope_business_summary_df


,scope,business_count,business_rate_pct
0,전체 업체,150346,100.00
1,Restaurants,52268,34.77
2,Food,27781,18.48
3,Restaurants와 Food 교집합,15472,10.29
4,Food Only,12309,8.19
5,Restaurants 또는 Food,64577,42.95


In [16]:
assert len(business_df) == 150_346

assert (
    business_df["is_restaurant"].sum()
    == 52_268
)

assert (
    business_df["is_food"].sum()
    == 27_781
)

assert (
    business_df[
        "is_restaurant_and_food"
    ].sum()
    == 15_472
)

assert (
    business_df["is_food_only"].sum()
    == 12_309
)

assert (
    business_df[
        "is_expanded_food"
    ].sum()
    == 64_577
)

print("Restaurants·Food 업체 범위 검증 통과")

Restaurants·Food 업체 범위 검증 통과


In [17]:
scope_business_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_scope_business_summary_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

In [18]:
# 6. Food-only 업체 추출
food_only_business_df = (
    business_df[
        business_df["is_food_only"]
    ]
    .copy()
    .reset_index(drop=True)
)

food_only_business_ids = set(
    food_only_business_df[
        "business_id"
    ]
)

print("Food-only 업체:", len(food_only_business_df))
print("Food-only 고유 ID:", len(food_only_business_ids))

assert len(food_only_business_df) == 12_309
assert food_only_business_df["business_id"].is_unique

food_only_business_df.head()

Food-only 업체: 12309
Food-only 고유 ID: 12309


,business_id,name,city,state,categories,category_list,is_restaurant,is_food,is_restaurant_and_food,is_food_only,is_expanded_food
0,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,Green Lane,PA,"Brewpubs, Breweries, Food","[Brewpubs, Breweries, Food]",False,True,False,True,True
1,JX4tUpd09YFchLBuI43lGw,Naked Cyber Cafe & Espresso Bar,Edmonton,AB,"Arts & Entertainment, Music Venues, Internet S...","[Arts & Entertainment, Music Venues, Internet ...",False,True,False,True,True
2,5BmQX4UVJY19mMtafMg7JA,Breadland Organic Whole Grain Bakery,Edmonton,AB,"Specialty Food, Bakeries, Food, Health Markets","[Specialty Food, Bakeries, Food, Health Markets]",False,True,False,True,True
3,0qNpTGTcqPwOLi2hADx4Xw,Charlie's Market,Tampa,FL,"Food, Grocery, Convenience Stores","[Food, Grocery, Convenience Stores]",False,True,False,True,True
4,txyXRytGjwOXvS8s4sc-WA,Smoothie King,Tucson,AZ,"Vitamins & Supplements, Ice Cream & Frozen Yog...","[Vitamins & Supplements, Ice Cream & Frozen Yo...",False,True,False,True,True


In [19]:
# 7. Food-only 세부 카테고리 집계
food_only_category_df = (
    food_only_business_df[
        [
            "business_id",
            "category_list"
        ]
    ]
    .explode(
        "category_list"
    )
    .rename(
        columns={
            "category_list": "category"
        }
    )
)

# 모든 업체에 공통으로 붙은 Food 카테고리는 제외
food_only_category_df = (
    food_only_category_df[
        (
            food_only_category_df[
                "category"
            ].notna()
        )
        & (
            food_only_category_df[
                "category"
            ] != ""
        )
        & (
            food_only_category_df[
                "category"
            ] != "Food"
        )
    ]
    .copy()
)

food_only_category_summary_df = (
    food_only_category_df
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        business_count=(
            "business_id",
            "nunique"
        )
    )
    .sort_values(
        "business_count",
        ascending=False
    )
    .reset_index(drop=True)
)

food_only_category_summary_df[
    "food_only_business_rate_pct"
] = (
    food_only_category_summary_df[
        "business_count"
    ]
    / len(food_only_business_df)
    * 100
).round(2)

food_only_category_summary_df.head(50)

,category,business_count,food_only_business_rate_pct
0,Shopping,2746,22.31
1,Coffee & Tea,2650,21.53
2,Grocery,2312,18.78
3,Specialty Food,2222,18.05
4,Ice Cream & Frozen Yogurt,1569,12.75
5,Convenience Stores,1384,11.24
6,Desserts,1348,10.95
7,Drugstores,1319,10.72
8,Bakeries,1261,10.24
9,Wine & Spirits,1259,10.23


In [20]:
food_only_category_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_only_category_summary_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

In [21]:
# 8. Food-only 업체 지역 분포
food_only_region_summary_df = (
    food_only_business_df
    .groupby(
        [
            "state",
            "city"
        ],
        as_index=False
    )
    .agg(
        business_count=(
            "business_id",
            "nunique"
        )
    )
    .sort_values(
        "business_count",
        ascending=False
    )
    .reset_index(drop=True)
)

food_only_region_summary_df.head(30)

,state,city,business_count
0,PA,Philadelphia,1221
1,FL,Tampa,705
2,AZ,Tucson,642
3,IN,Indianapolis,618
4,TN,Nashville,542
5,LA,New Orleans,538
6,AB,Edmonton,499
7,NV,Reno,435
8,MO,Saint Louis,316
9,CA,Santa Barbara,312


In [22]:
food_only_region_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_only_region_summary_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

In [23]:
# 9. Food-only 리뷰 집계
if (
    FOOD_ONLY_MONTHLY_CACHE_PATH.exists()
    and FOOD_ONLY_BUSINESS_REVIEW_CACHE_PATH.exists()
):
    food_only_monthly_df = pd.read_parquet(
        FOOD_ONLY_MONTHLY_CACHE_PATH
    )

    food_only_business_review_count_df = (
        pd.read_parquet(
            FOOD_ONLY_BUSINESS_REVIEW_CACHE_PATH
        )
    )

    print("기존 Food-only 집계 캐시를 불러왔습니다.")

else:
    monthly_parts = []
    business_review_parts = []

    for chunk_number, chunk_df in enumerate(
        pd.read_json(
            REVIEW_JSON_PATH,
            lines=True,
            chunksize=250_000
        ),
        start=1
    ):
        food_review_df = (
            chunk_df[
                chunk_df[
                    "business_id"
                ].isin(
                    food_only_business_ids
                )
            ]
            [
                [
                    "user_id",
                    "business_id",
                    "date"
                ]
            ]
            .copy()
        )

        if food_review_df.empty:
            print(
                f"{chunk_number}번째 청크: "
                "Food-only 리뷰 없음"
            )
            continue

        food_review_df["date"] = pd.to_datetime(
            food_review_df["date"],
            errors="coerce"
        )

        food_review_df = (
            food_review_df
            .dropna(
                subset=[
                    "user_id",
                    "business_id",
                    "date"
                ]
            )
        )

        food_review_df["year"] = (
            food_review_df["date"]
            .dt.year
            .astype("int16")
        )

        food_review_df["month"] = (
            food_review_df["date"]
            .dt.month
            .astype("int8")
        )

        chunk_monthly_df = (
            food_review_df
            .groupby(
                [
                    "user_id",
                    "year",
                    "month"
                ],
                as_index=False
            )
            .size()
            .rename(
                columns={
                    "size": "review_count"
                }
            )
        )

        chunk_business_review_df = (
            food_review_df
            .groupby(
                "business_id",
                as_index=False
            )
            .size()
            .rename(
                columns={
                    "size": "review_count"
                }
            )
        )

        monthly_parts.append(
            chunk_monthly_df
        )

        business_review_parts.append(
            chunk_business_review_df
        )

        print(
            f"{chunk_number}번째 청크 완료: "
            f"{len(food_review_df):,}건"
        )

        del food_review_df
        del chunk_monthly_df
        del chunk_business_review_df
        gc.collect()

    food_only_monthly_df = (
        pd.concat(
            monthly_parts,
            ignore_index=True
        )
        .groupby(
            [
                "user_id",
                "year",
                "month"
            ],
            as_index=False
        )
        ["review_count"]
        .sum()
    )

    food_only_business_review_count_df = (
        pd.concat(
            business_review_parts,
            ignore_index=True
        )
        .groupby(
            "business_id",
            as_index=False
        )
        ["review_count"]
        .sum()
    )

    food_only_monthly_df.to_parquet(
        FOOD_ONLY_MONTHLY_CACHE_PATH,
        index=False
    )

    food_only_business_review_count_df.to_parquet(
        FOOD_ONLY_BUSINESS_REVIEW_CACHE_PATH,
        index=False
    )

    print("Food-only 리뷰 집계 캐시 저장 완료")

1번째 청크 완료: 15,462건
2번째 청크 완료: 15,441건
3번째 청크 완료: 15,843건
4번째 청크 완료: 12,258건
5번째 청크 완료: 13,084건
6번째 청크 완료: 13,734건
7번째 청크 완료: 12,018건
8번째 청크 완료: 13,693건
9번째 청크 완료: 14,507건
10번째 청크 완료: 12,906건
11번째 청크 완료: 14,103건
12번째 청크 완료: 15,332건
13번째 청크 완료: 15,441건
14번째 청크 완료: 16,799건
15번째 청크 완료: 13,543건
16번째 청크 완료: 13,795건
17번째 청크 완료: 14,286건
18번째 청크 완료: 13,704건
19번째 청크 완료: 15,034건
20번째 청크 완료: 15,446건
21번째 청크 완료: 13,729건
22번째 청크 완료: 14,619건
23번째 청크 완료: 14,403건
24번째 청크 완료: 13,404건
25번째 청크 완료: 14,512건
26번째 청크 완료: 14,604건
27번째 청크 완료: 13,968건
28번째 청크 완료: 14,282건
Food-only 리뷰 집계 캐시 저장 완료


In [24]:
# 10. Food-only 리뷰 집계 결과
food_only_review_count = int(
    food_only_monthly_df[
        "review_count"
    ].sum()
)

food_only_unique_users = int(
    food_only_monthly_df[
        "user_id"
    ].nunique()
)

food_only_reviewed_businesses = int(
    food_only_business_review_count_df[
        "business_id"
    ].nunique()
)

food_only_review_per_business = (
    food_only_review_count
    / food_only_reviewed_businesses
)

food_only_review_per_user = (
    food_only_review_count
    / food_only_unique_users
)

print(
    "Food-only 리뷰:",
    f"{food_only_review_count:,}"
)

print(
    "Food-only 리뷰 사용자:",
    f"{food_only_unique_users:,}"
)

print(
    "리뷰가 있는 Food-only 업체:",
    f"{food_only_reviewed_businesses:,}"
)

print(
    "업체당 평균 리뷰:",
    f"{food_only_review_per_business:.2f}"
)

print(
    "사용자당 평균 리뷰:",
    f"{food_only_review_per_user:.2f}"
)

Food-only 리뷰: 399,950
Food-only 리뷰 사용자: 216,746
리뷰가 있는 Food-only 업체: 12,309
업체당 평균 리뷰: 32.49
사용자당 평균 리뷰: 1.85


In [25]:
# 연도별 리뷰:
food_only_year_summary_df = (
    food_only_monthly_df
    .groupby(
        "year",
        as_index=False
    )
    .agg(
        review_count=(
            "review_count",
            "sum"
        ),
        unique_users=(
            "user_id",
            "nunique"
        )
    )
    .sort_values("year")
    .reset_index(drop=True)
)

food_only_year_summary_df

,year,review_count,unique_users
0,2005,28,18
1,2006,164,100
2,2007,749,391
3,2008,2995,1282
4,2009,4171,2369
5,2010,8151,4573
6,2011,13746,7770
7,2012,16855,9710
8,2013,22786,13358
9,2014,28711,18661


In [26]:
food_only_year_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_only_year_review_summary_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

In [27]:
# 11. 기존 Restaurant 리뷰 월간 집계
if RESTAURANT_MONTHLY_CACHE_PATH.exists():
    restaurant_monthly_df = pd.read_parquet(
        RESTAURANT_MONTHLY_CACHE_PATH
    )

    print("기존 Restaurant 월간 캐시를 불러왔습니다.")

else:
    restaurant_review_df = pd.read_parquet(
        RESTAURANT_REVIEW_PATH,
        columns=[
            "user_id",
            "date"
        ]
    )

    restaurant_review_df["date"] = pd.to_datetime(
        restaurant_review_df["date"],
        errors="coerce"
    )

    restaurant_review_df = (
        restaurant_review_df
        .dropna(
            subset=[
                "user_id",
                "date"
            ]
        )
    )

    restaurant_review_df["year"] = (
        restaurant_review_df["date"]
        .dt.year
        .astype("int16")
    )

    restaurant_review_df["month"] = (
        restaurant_review_df["date"]
        .dt.month
        .astype("int8")
    )

    restaurant_monthly_df = (
        restaurant_review_df
        .groupby(
            [
                "user_id",
                "year",
                "month"
            ],
            as_index=False
        )
        .size()
        .rename(
            columns={
                "size": "review_count"
            }
        )
    )

    restaurant_monthly_df.to_parquet(
        RESTAURANT_MONTHLY_CACHE_PATH,
        index=False
    )

    del restaurant_review_df
    gc.collect()

    print("Restaurant 월간 캐시 저장 완료")

Restaurant 월간 캐시 저장 완료


In [28]:
# 12. Restaurants와 확장 범위 리뷰 비교
expanded_monthly_df = (
    pd.concat(
        [
            restaurant_monthly_df,
            food_only_monthly_df
        ],
        ignore_index=True
    )
    .groupby(
        [
            "user_id",
            "year",
            "month"
        ],
        as_index=False
    )
    ["review_count"]
    .sum()
)

restaurant_review_count = int(
    restaurant_monthly_df[
        "review_count"
    ].sum()
)

expanded_review_count = int(
    expanded_monthly_df[
        "review_count"
    ].sum()
)

restaurant_unique_users = int(
    restaurant_monthly_df[
        "user_id"
    ].nunique()
)

expanded_unique_users = int(
    expanded_monthly_df[
        "user_id"
    ].nunique()
)

scope_review_summary_df = pd.DataFrame(
    {
        "scope": [
            "Restaurants",
            "Food Only 추가분",
            "Restaurants 또는 Food"
        ],
        "business_count": [
            52_268,
            12_309,
            64_577
        ],
        "review_count": [
            restaurant_review_count,
            food_only_review_count,
            expanded_review_count
        ],
        "unique_users": [
            restaurant_unique_users,
            food_only_unique_users,
            expanded_unique_users
        ]
    }
)

scope_review_summary_df

,scope,business_count,review_count,unique_users
0,Restaurants,52268,4724471,1445990
1,Food Only 추가분,12309,399950,216746
2,Restaurants 또는 Food,64577,5124421,1504895


In [29]:
review_increase_rate_pct = (
    food_only_review_count
    / restaurant_review_count
    * 100
)

user_increase_count = (
    expanded_unique_users
    - restaurant_unique_users
)

user_increase_rate_pct = (
    user_increase_count
    / restaurant_unique_users
    * 100
)

print(
    "리뷰 증가율:",
    f"{review_increase_rate_pct:.2f}%"
)

print(
    "고유 사용자 증가:",
    f"{user_increase_count:,}명"
)

print(
    "고유 사용자 증가율:",
    f"{user_increase_rate_pct:.2f}%"
)

리뷰 증가율: 8.47%
고유 사용자 증가: 58,905명
고유 사용자 증가율: 4.07%


In [30]:
scope_review_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_scope_review_summary_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# 13. 연간 코호트 생성 함수
def prepare_activity_tables(
    monthly_df: pd.DataFrame
):
    annual_activity_df = (
        monthly_df
        .groupby(
            [
                "user_id",
                "year"
            ],
            as_index=False
        )
        .agg(
            review_count=(
                "review_count",
                "sum"
            ),
            active_months=(
                "month",
                "nunique"
            )
        )
    )

    second_half_activity_df = (
        monthly_df[
            monthly_df["month"] >= 7
        ]
        .groupby(
            [
                "user_id",
                "year"
            ],
            as_index=False
        )
        .agg(
            second_half_review_count=(
                "review_count",
                "sum"
            )
        )
    )

    return (
        annual_activity_df,
        second_half_activity_df
    )

In [33]:
def build_annual_cohort(
    annual_activity_df: pd.DataFrame,
    second_half_activity_df: pd.DataFrame,
    selection_year: int,
    minimum_review_count: int = 10,
    minimum_active_months: int = 3,
    minimum_continuity_reviews: int = 1
):
    observation_year = (
        selection_year + 1
    )

    target_year = (
        selection_year + 2
    )

    baseline_df = (
        annual_activity_df[
            (
                annual_activity_df["year"]
                == selection_year
            )
            & (
                annual_activity_df["review_count"]
                >= minimum_review_count
            )
            & (
                annual_activity_df["active_months"]
                >= minimum_active_months
            )
        ]
        [
            [
                "user_id",
                "review_count",
                "active_months"
            ]
        ]
        .rename(
            columns={
                "review_count":
                    "baseline_review_count",
                "active_months":
                    "baseline_active_months"
            }
        )
    )

    continuity_df = (
        second_half_activity_df[
            (
                second_half_activity_df["year"]
                == observation_year
            )
            & (
                second_half_activity_df[
                    "second_half_review_count"
                ]
                >= minimum_continuity_reviews
            )
        ]
        [
            [
                "user_id",
                "second_half_review_count"
            ]
        ]
    )

    target_df = (
        annual_activity_df[
            annual_activity_df["year"]
            == target_year
        ]
        [
            [
                "user_id",
                "review_count"
            ]
        ]
        .rename(
            columns={
                "review_count":
                    "target_review_count"
            }
        )
    )

    cohort_df = (
        baseline_df
        .merge(
            continuity_df,
            on="user_id",
            how="inner",
            validate="one_to_one"
        )
        .merge(
            target_df,
            on="user_id",
            how="left",
            validate="one_to_one"
        )
    )

    cohort_df["target_review_count"] = (
        cohort_df["target_review_count"]
        .fillna(0)
        .astype("int32")
    )

    cohort_df["churn"] = (
        cohort_df["target_review_count"]
        == 0
    ).astype("int8")

    cohort_df["selection_year"] = (
        selection_year
    )

    cohort_df["observation_year"] = (
        observation_year
    )

    cohort_df["target_year"] = (
        target_year
    )

    assert cohort_df["user_id"].is_unique
    assert cohort_df["churn"].isin([0, 1]).all()

    return cohort_df

In [34]:
# 14. Restaurant·확장 활동 테이블 생성
(
    restaurant_annual_activity_df,
    restaurant_second_half_activity_df
) = prepare_activity_tables(
    restaurant_monthly_df
)

(
    expanded_annual_activity_df,
    expanded_second_half_activity_df
) = prepare_activity_tables(
    expanded_monthly_df
)

In [35]:
# 15. 연도별 롤링 코호트 비교
selection_years = [
    2013,
    2014,
    2015,
    2016,
    2017
]

cohort_results = {}
cohort_summary_rows = []

for scope_name, annual_df, second_half_df in [
    (
        "Restaurants",
        restaurant_annual_activity_df,
        restaurant_second_half_activity_df
    ),
    (
        "Restaurants 또는 Food",
        expanded_annual_activity_df,
        expanded_second_half_activity_df
    )
]:
    for selection_year in selection_years:
        cohort_df = build_annual_cohort(
            annual_activity_df=annual_df,
            second_half_activity_df=second_half_df,
            selection_year=selection_year
        )

        cohort_results[
            (
                scope_name,
                selection_year
            )
        ] = cohort_df

        cohort_summary_rows.append(
            {
                "scope": scope_name,
                "selection_year": selection_year,
                "observation_year":
                    selection_year + 1,
                "target_year":
                    selection_year + 2,
                "candidate_users":
                    len(cohort_df),
                "churn_users":
                    int(
                        cohort_df[
                            "churn"
                        ].sum()
                    )
            }
        )

rolling_cohort_summary_df = pd.DataFrame(
    cohort_summary_rows
)

rolling_cohort_summary_df[
    "retained_users"
] = (
    rolling_cohort_summary_df[
        "candidate_users"
    ]
    - rolling_cohort_summary_df[
        "churn_users"
    ]
)

rolling_cohort_summary_df[
    "churn_rate_pct"
] = (
    rolling_cohort_summary_df[
        "churn_users"
    ]
    / rolling_cohort_summary_df[
        "candidate_users"
    ]
    * 100
).round(2)

rolling_cohort_summary_df

,scope,selection_year,observation_year,target_year,candidate_users,churn_users,retained_users,churn_rate_pct
0,Restaurants,2013,2014,2015,2234,337,1897,15.09
1,Restaurants,2014,2015,2016,2762,444,2318,16.08
2,Restaurants,2015,2016,2017,3320,517,2803,15.57
3,Restaurants,2016,2017,2018,3537,510,3027,14.42
4,Restaurants,2017,2018,2019,3908,606,3302,15.51
5,Restaurants 또는 Food,2013,2014,2015,2424,379,2045,15.64
6,Restaurants 또는 Food,2014,2015,2016,3041,488,2553,16.05
7,Restaurants 또는 Food,2015,2016,2017,3680,579,3101,15.73
8,Restaurants 또는 Food,2016,2017,2018,3891,553,3338,14.21
9,Restaurants 또는 Food,2017,2018,2019,4350,707,3643,16.25


In [36]:
# 16. 기존 2017 코호트 재현 검증
restaurant_2017_cohort_df = (
    cohort_results[
        (
            "Restaurants",
            2017
        )
    ]
)

assert (
    len(
        restaurant_2017_cohort_df
    )
    == 3_908
)

assert (
    restaurant_2017_cohort_df[
        "churn"
    ].sum()
    == 606
)

print("기존 2017 Restaurants 코호트 재현 통과")

기존 2017 Restaurants 코호트 재현 통과


In [37]:
expanded_2017_cohort_df = (
    cohort_results[
        (
            "Restaurants 또는 Food",
            2017
        )
    ]
)

print(
    "기존 코호트:",
    f"{len(restaurant_2017_cohort_df):,}명"
)

print(
    "확장 코호트:",
    f"{len(expanded_2017_cohort_df):,}명"
)

print(
    "코호트 증가:",
    f"{len(expanded_2017_cohort_df) - len(restaurant_2017_cohort_df):,}명"
)

print(
    "확장 코호트 이탈자:",
    f"{expanded_2017_cohort_df['churn'].sum():,}명"
)

기존 코호트: 3,908명
확장 코호트: 4,350명
코호트 증가: 442명
확장 코호트 이탈자: 707명


In [38]:
# 17. 기존 사용자 라벨 변화 확인
label_comparison_df = (
    restaurant_2017_cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .rename(
        columns={
            "churn":
                "restaurant_churn"
        }
    )
    .merge(
        expanded_2017_cohort_df[
            [
                "user_id",
                "churn"
            ]
        ]
        .rename(
            columns={
                "churn":
                    "expanded_churn"
            }
        ),
        on="user_id",
        how="outer",
        indicator=True
    )
)

membership_summary_df = (
    label_comparison_df[
        "_merge"
    ]
    .value_counts()
    .rename_axis(
        "membership"
    )
    .reset_index(
        name="users"
    )
)

membership_summary_df


,membership,users
0,both,3908
1,right_only,442
2,left_only,0


In [39]:
common_user_label_df = (
    label_comparison_df[
        label_comparison_df["_merge"]
        == "both"
    ]
    .copy()
)

label_transition_df = pd.crosstab(
    common_user_label_df[
        "restaurant_churn"
    ],
    common_user_label_df[
        "expanded_churn"
    ]
).reset_index()

label_transition_df

expanded_churn,restaurant_churn,0,1
0,0.0,3302,0
1,1.0,19,587


In [40]:
restaurant_churn_but_expanded_retained_df = (
    common_user_label_df[
        (
            common_user_label_df[
                "restaurant_churn"
            ]
            == 1
        )
        & (
            common_user_label_df[
                "expanded_churn"
            ]
            == 0
        )
    ]
    .copy()
)

print(
    "Restaurants 기준 이탈이지만 "
    "확장 기준 유지:",
    len(
        restaurant_churn_but_expanded_retained_df
    )
)

Restaurants 기준 이탈이지만 확장 기준 유지: 19


In [ ]:
# 18. 결과 저장
rolling_cohort_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_scope_rolling_cohort_summary_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

membership_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "food_scope_membership_comparison_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

label_transition_df.to_csv(
    REPORT_TABLE_DIR
    / "food_scope_label_transition_v01.csv",
    index=False,
    encoding="utf-8-sig"
)

In [42]:
# 19. 최종 검토용 요약
restaurant_2017_count = len(
    restaurant_2017_cohort_df
)

expanded_2017_count = len(
    expanded_2017_cohort_df
)

cohort_increase_count = (
    expanded_2017_count
    - restaurant_2017_count
)

cohort_increase_rate_pct = (
    cohort_increase_count
    / restaurant_2017_count
    * 100
)

validation_result_df = pd.DataFrame(
    {
        "metric": [
            "Food-only 추가 업체",
            "Food-only 추가 리뷰",
            "리뷰 증가율",
            "고유 사용자 증가",
            "2017 코호트 증가",
            "2017 코호트 증가율",
            "기존 이탈→확장 유지 사용자"
        ],
        "value": [
            len(food_only_business_df),
            food_only_review_count,
            round(
                review_increase_rate_pct,
                2
            ),
            user_increase_count,
            cohort_increase_count,
            round(
                cohort_increase_rate_pct,
                2
            ),
            len(
                restaurant_churn_but_expanded_retained_df
            )
        ]
    }
)

validation_result_df

,metric,value
0,Food-only 추가 업체,12309.00
1,Food-only 추가 리뷰,399950.00
2,리뷰 증가율,8.47
3,고유 사용자 증가,58905.00
4,2017 코호트 증가,442.00
5,2017 코호트 증가율,11.31
6,기존 이탈→확장 유지 사용자,19.00


In [43]:
restaurant_2017_count = len(
    restaurant_2017_cohort_df
)

expanded_2017_count = len(
    expanded_2017_cohort_df
)

cohort_increase_count = (
    expanded_2017_count
    - restaurant_2017_count
)

cohort_increase_rate_pct = (
    cohort_increase_count
    / restaurant_2017_count
    * 100
)

validation_result_df = pd.DataFrame(
    {
        "metric": [
            "Food-only 추가 업체",
            "Food-only 추가 리뷰",
            "리뷰 증가율",
            "고유 사용자 증가",
            "2017 코호트 증가",
            "2017 코호트 증가율",
            "기존 이탈→확장 유지 사용자"
        ],
        "value": [
            len(food_only_business_df),
            food_only_review_count,
            round(
                review_increase_rate_pct,
                2
            ),
            user_increase_count,
            cohort_increase_count,
            round(
                cohort_increase_rate_pct,
                2
            ),
            len(
                restaurant_churn_but_expanded_retained_df
            )
        ]
    }
)

validation_result_df

,metric,value
0,Food-only 추가 업체,12309.00
1,Food-only 추가 리뷰,399950.00
2,리뷰 증가율,8.47
3,고유 사용자 증가,58905.00
4,2017 코호트 증가,442.00
5,2017 코호트 증가율,11.31
6,기존 이탈→확장 유지 사용자,19.00


In [44]:
food_only_category_summary_df.head(50)

,category,business_count,food_only_business_rate_pct
0,Shopping,2746,22.31
1,Coffee & Tea,2650,21.53
2,Grocery,2312,18.78
3,Specialty Food,2222,18.05
4,Ice Cream & Frozen Yogurt,1569,12.75
5,Convenience Stores,1384,11.24
6,Desserts,1348,10.95
7,Drugstores,1319,10.72
8,Bakeries,1261,10.24
9,Wine & Spirits,1259,10.23


In [45]:
# 미식 방문형 추가 카테고리

CULINARY_VISIT_CATEGORIES = {
    "Cafes",
    "Coffee & Tea",
    "Ice Cream & Frozen Yogurt",
    "Desserts",
    "Bakeries",
    "Juice Bars & Smoothies",
    "Donuts",
    "Cupcakes",
    "Food Trucks",
    "Bubble Tea",
    "Shaved Ice"
}

ALCOHOL_VISIT_CATEGORIES = {
    "Bars",
    "Breweries",
    "Wineries"
}


def contains_target_category(
    category_list,
    target_categories
):
    return bool(
        set(category_list)
        & target_categories
    )


food_only_business_df[
    "is_culinary_visit"
] = (
    food_only_business_df[
        "category_list"
    ]
    .apply(
        lambda categories:
            contains_target_category(
                categories,
                CULINARY_VISIT_CATEGORIES
            )
    )
)

food_only_business_df[
    "is_alcohol_visit"
] = (
    food_only_business_df[
        "category_list"
    ]
    .apply(
        lambda categories:
            contains_target_category(
                categories,
                ALCOHOL_VISIT_CATEGORIES
            )
    )
)

In [46]:
# 1. 카페·디저트 등 미식 방문형
culinary_visit_business_df = (
    food_only_business_df[
        food_only_business_df[
            "is_culinary_visit"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# 2. 주류 방문 장소
alcohol_visit_business_df = (
    food_only_business_df[
        (
            food_only_business_df[
                "is_alcohol_visit"
            ]
        )
        & (
            ~food_only_business_df[
                "is_culinary_visit"
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

# 3. 미식 방문형 + 주류 방문형
culinary_with_alcohol_business_df = (
    food_only_business_df[
        (
            food_only_business_df[
                "is_culinary_visit"
            ]
        )
        | (
            food_only_business_df[
                "is_alcohol_visit"
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

In [47]:
curated_scope_summary_df = pd.DataFrame(
    {
        "scope": [
            "Restaurants",
            "미식 방문형 추가 업체",
            "주류 방문형 추가 업체",
            "Restaurants + 미식 방문형",
            "Restaurants + 미식·주류 방문형",
            "Restaurants 또는 Food 전체"
        ],
        "business_count": [
            52_268,
            len(
                culinary_visit_business_df
            ),
            len(
                alcohol_visit_business_df
            ),
            (
                52_268
                + len(
                    culinary_visit_business_df
                )
            ),
            (
                52_268
                + len(
                    culinary_with_alcohol_business_df
                )
            ),
            64_577
        ]
    }
)

curated_scope_summary_df[
    "business_rate_pct"
] = (
    curated_scope_summary_df[
        "business_count"
    ]
    / 150_346
    * 100
).round(2)

curated_scope_summary_df

,scope,business_count,business_rate_pct
0,Restaurants,52268,34.77
1,미식 방문형 추가 업체,5888,3.92
2,주류 방문형 추가 업체,871,0.58
3,Restaurants + 미식 방문형,58156,38.68
4,Restaurants + 미식·주류 방문형,59027,39.26
5,Restaurants 또는 Food 전체,64577,42.95


In [ ]:
# 이제 5,888개 미식 방문형 업체가 실제로 리뷰와 코호트를 얼마나 늘리는지 계산해야 해.

# 기존 food_only_monthly_df에는 business_id가 없기 때문에 5,888개만 다시 필터링할 수 없어. 
# 원본 Review JSON을 한 번 다시 읽고 미식 방문형 월간 집계를 캐시해야 해.

In [49]:
# 1. 경로 설정
CULINARY_MONTHLY_CACHE_PATH = (
    VALIDATION_CACHE_DIR
    / "culinary_visit_monthly_activity_v01.parquet"
)

CULINARY_BUSINESS_REVIEW_CACHE_PATH = (
    VALIDATION_CACHE_DIR
    / "culinary_visit_business_review_counts_v01.parquet"
)

culinary_visit_business_ids = set(
    culinary_visit_business_df[
        "business_id"
    ]
)

assert len(
    culinary_visit_business_ids
) == 5_888

In [ ]:
# 2. 미식 방문형 리뷰 집계
if (
    CULINARY_MONTHLY_CACHE_PATH.exists()
    and CULINARY_BUSINESS_REVIEW_CACHE_PATH.exists()
):
    culinary_monthly_df = pd.read_parquet(
        CULINARY_MONTHLY_CACHE_PATH
    )

    culinary_business_review_count_df = (
        pd.read_parquet(
            CULINARY_BUSINESS_REVIEW_CACHE_PATH
        )
    )

    print("미식 방문형 리뷰 캐시를 불러왔습니다.")

else:
    monthly_parts = []
    business_review_parts = []

    for chunk_number, chunk_df in enumerate(
        pd.read_json(
            REVIEW_JSON_PATH,
            lines=True,
            chunksize=250_000
        ),
        start=1
    ):
        culinary_review_df = (
            chunk_df[
                chunk_df[
                    "business_id"
                ].isin(
                    culinary_visit_business_ids
                )
            ]
            [
                [
                    "user_id",
                    "business_id",
                    "date"
                ]
            ]
            .copy()
        )

        if culinary_review_df.empty:
            continue

        culinary_review_df["date"] = (
            pd.to_datetime(
                culinary_review_df["date"],
                errors="coerce"
            )
        )

        culinary_review_df = (
            culinary_review_df
            .dropna(
                subset=[
                    "user_id",
                    "business_id",
                    "date"
                ]
            )
        )

        culinary_review_df["year"] = (
            culinary_review_df["date"]
            .dt.year
            .astype("int16")
        )

        culinary_review_df["month"] = (
            culinary_review_df["date"]
            .dt.month
            .astype("int8")
        )

        chunk_monthly_df = (
            culinary_review_df
            .groupby(
                [
                    "user_id",
                    "year",
                    "month"
                ],
                as_index=False
            )
            .size()
            .rename(
                columns={
                    "size": "review_count"
                }
            )
        )

        chunk_business_df = (
            culinary_review_df
            .groupby(
                "business_id",
                as_index=False
            )
            .size()
            .rename(
                columns={
                    "size": "review_count"
                }
            )
        )

        monthly_parts.append(
            chunk_monthly_df
        )

        business_review_parts.append(
            chunk_business_df
        )

        print(
            f"{chunk_number}번째 청크 완료: "
            f"{len(culinary_review_df):,}건"
        )

    culinary_monthly_df = (
        pd.concat(
            monthly_parts,
            ignore_index=True
        )
        .groupby(
            [
                "user_id",
                "year",
                "month"
            ],
            as_index=False
        )
        ["review_count"]
        .sum()
    )

    culinary_business_review_count_df = (
        pd.concat(
            business_review_parts,
            ignore_index=True
        )
        .groupby(
            "business_id",
            as_index=False
        )
        ["review_count"]
        .sum()
    )

    culinary_monthly_df.to_parquet(
        CULINARY_MONTHLY_CACHE_PATH,
        index=False
    )

    culinary_business_review_count_df.to_parquet(
        CULINARY_BUSINESS_REVIEW_CACHE_PATH,
        index=False
    )

    print("미식 방문형 리뷰 캐시 저장 완료")

1번째 청크 완료: 8,131건
2번째 청크 완료: 7,986건
3번째 청크 완료: 8,393건
4번째 청크 완료: 7,414건
5번째 청크 완료: 7,975건
6번째 청크 완료: 7,931건
7번째 청크 완료: 6,398건
8번째 청크 완료: 7,150건
9번째 청크 완료: 8,042건
10번째 청크 완료: 7,890건
11번째 청크 완료: 8,244건
12번째 청크 완료: 9,333건
13번째 청크 완료: 9,535건
14번째 청크 완료: 10,139건
15번째 청크 완료: 8,061건
16번째 청크 완료: 7,714건
17번째 청크 완료: 7,859건
18번째 청크 완료: 7,094건
19번째 청크 완료: 8,068건
20번째 청크 완료: 8,867건
21번째 청크 완료: 8,252건
22번째 청크 완료: 8,604건
23번째 청크 완료: 7,739건
24번째 청크 완료: 6,875건
25번째 청크 완료: 7,446건
26번째 청크 완료: 8,552건
27번째 청크 완료: 8,157건
28번째 청크 완료: 7,944건
미식 방문형 리뷰 캐시 저장 완료


In [51]:
# 3. Restaurants와 결합
restaurant_culinary_monthly_df = (
    pd.concat(
        [
            restaurant_monthly_df,
            culinary_monthly_df
        ],
        ignore_index=True
    )
    .groupby(
        [
            "user_id",
            "year",
            "month"
        ],
        as_index=False
    )
    ["review_count"]
    .sum()
)

In [52]:
# 4. 리뷰 증가량 확인
culinary_review_count = int(
    culinary_monthly_df[
        "review_count"
    ].sum()
)

culinary_unique_users = int(
    culinary_monthly_df[
        "user_id"
    ].nunique()
)

restaurant_culinary_review_count = int(
    restaurant_culinary_monthly_df[
        "review_count"
    ].sum()
)

culinary_review_increase_rate_pct = (
    culinary_review_count
    / restaurant_review_count
    * 100
)

print(
    "미식 방문형 추가 리뷰:",
    f"{culinary_review_count:,}"
)

print(
    "미식 방문형 리뷰 사용자:",
    f"{culinary_unique_users:,}"
)

print(
    "확장 후 전체 리뷰:",
    f"{restaurant_culinary_review_count:,}"
)

print(
    "리뷰 증가율:",
    f"{culinary_review_increase_rate_pct:.2f}%"
)

미식 방문형 추가 리뷰: 225,793
미식 방문형 리뷰 사용자: 140,133
확장 후 전체 리뷰: 4,950,264
리뷰 증가율: 4.78%


In [53]:
restaurant_user_ids = set(
    restaurant_monthly_df[
        "user_id"
    ].unique()
)

culinary_user_ids = set(
    culinary_monthly_df[
        "user_id"
    ].unique()
)

new_culinary_user_ids = (
    culinary_user_ids
    - restaurant_user_ids
)

print(
    "미식 방문형 리뷰 사용자:",
    f"{len(culinary_user_ids):,}명"
)

print(
    "기존 Restaurants와 중복:",
    f"{len(culinary_user_ids & restaurant_user_ids):,}명"
)

print(
    "실제 신규 사용자:",
    f"{len(new_culinary_user_ids):,}명"
)

미식 방문형 리뷰 사용자: 140,133명
기존 Restaurants와 중복: 107,057명
실제 신규 사용자: 33,076명


In [54]:
# 1. 미식 확장 활동 테이블 생성
(
    restaurant_culinary_annual_df,
    restaurant_culinary_second_half_df
) = prepare_activity_tables(
    restaurant_culinary_monthly_df
)

In [55]:
# 2. 2017 코호트 생성
restaurant_culinary_2017_cohort_df = (
    build_annual_cohort(
        annual_activity_df=
            restaurant_culinary_annual_df,
        second_half_activity_df=
            restaurant_culinary_second_half_df,
        selection_year=2017
    )
)

print(
    "Restaurants 코호트:",
    f"{len(restaurant_2017_cohort_df):,}명"
)

print(
    "미식 확장 코호트:",
    f"{len(restaurant_culinary_2017_cohort_df):,}명"
)

print(
    "코호트 증가:",
    f"{len(restaurant_culinary_2017_cohort_df) - len(restaurant_2017_cohort_df):,}명"
)

print(
    "미식 확장 이탈자:",
    f"{restaurant_culinary_2017_cohort_df['churn'].sum():,}명"
)

print(
    "미식 확장 이탈률:",
    f"{restaurant_culinary_2017_cohort_df['churn'].mean() * 100:.2f}%"
)

Restaurants 코호트: 3,908명
미식 확장 코호트: 4,157명
코호트 증가: 249명
미식 확장 이탈자: 670명
미식 확장 이탈률: 16.12%


In [56]:
# 3. 기존 이탈 라벨 변화
culinary_label_comparison_df = (
    restaurant_2017_cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .rename(
        columns={
            "churn":
                "restaurant_churn"
        }
    )
    .merge(
        restaurant_culinary_2017_cohort_df[
            [
                "user_id",
                "churn"
            ]
        ]
        .rename(
            columns={
                "churn":
                    "culinary_churn"
            }
        ),
        on="user_id",
        how="outer",
        indicator=True
    )
)

pd.crosstab(
    culinary_label_comparison_df[
        "restaurant_churn"
    ],
    culinary_label_comparison_df[
        "culinary_churn"
    ],
    dropna=False
)

culinary_churn,0,1
restaurant_churn,,
0.0,3302,0
1.0,10,596
NaN,175,74


In [58]:
def build_rolling_cohorts(
    annual_activity_df: pd.DataFrame,
    second_half_activity_df: pd.DataFrame,
    selection_years: list,
    scope_name: str
):
    cohort_list = []
    summary_rows = []

    for selection_year in selection_years:
        cohort_df = build_annual_cohort(
            annual_activity_df=
                annual_activity_df,
            second_half_activity_df=
                second_half_activity_df,
            selection_year=
                selection_year
        )

        cohort_df["scope"] = (
            scope_name
        )

        cohort_list.append(
            cohort_df
        )

        candidate_users = len(
            cohort_df
        )

        churn_users = int(
            cohort_df[
                "churn"
            ].sum()
        )

        retained_users = (
            candidate_users
            - churn_users
        )

        churn_rate_pct = (
            churn_users
            / candidate_users
            * 100
            if candidate_users > 0
            else 0
        )

        summary_rows.append(
            {
                "scope":
                    scope_name,
                "selection_year":
                    selection_year,
                "observation_year":
                    selection_year + 1,
                "target_year":
                    selection_year + 2,
                "candidate_users":
                    candidate_users,
                "churn_users":
                    churn_users,
                "retained_users":
                    retained_users,
                "churn_rate_pct":
                    round(
                        churn_rate_pct,
                        2
                    )
            }
        )

    rolling_cohort_df = pd.concat(
        cohort_list,
        ignore_index=True
    )

    rolling_summary_df = pd.DataFrame(
        summary_rows
    )

    return (
        rolling_cohort_df,
        rolling_summary_df
    )

In [59]:
# 2. 비교 연도 설정
comparison_selection_years = [
    2013,
    2014,
    2015,
    2016,
    2017
]
# 3. Restaurants 롤링 코호트 생성
(
    restaurant_rolling_df,
    restaurant_rolling_summary_df
) = build_rolling_cohorts(
    annual_activity_df=
        restaurant_annual_activity_df,
    second_half_activity_df=
        restaurant_second_half_activity_df,
    selection_years=
        comparison_selection_years,
    scope_name=
        "Restaurants"
)

restaurant_rolling_summary_df

,scope,selection_year,observation_year,target_year,candidate_users,churn_users,retained_users,churn_rate_pct
0,Restaurants,2013,2014,2015,2234,337,1897,15.09
1,Restaurants,2014,2015,2016,2762,444,2318,16.08
2,Restaurants,2015,2016,2017,3320,517,2803,15.57
3,Restaurants,2016,2017,2018,3537,510,3027,14.42
4,Restaurants,2017,2018,2019,3908,606,3302,15.51


In [60]:
# 4. Restaurants + 미식 방문형 롤링 코호트
(
    restaurant_culinary_rolling_df,
    restaurant_culinary_rolling_summary_df
) = build_rolling_cohorts(
    annual_activity_df=
        restaurant_culinary_annual_df,
    second_half_activity_df=
        restaurant_culinary_second_half_df,
    selection_years=
        comparison_selection_years,
    scope_name=
        "Restaurants + 미식 방문형"
)

restaurant_culinary_rolling_summary_df

,scope,selection_year,observation_year,target_year,candidate_users,churn_users,retained_users,churn_rate_pct
0,Restaurants + 미식 방문형,2013,2014,2015,2324,363,1961,15.62
1,Restaurants + 미식 방문형,2014,2015,2016,2917,461,2456,15.80
2,Restaurants + 미식 방문형,2015,2016,2017,3513,557,2956,15.86
3,Restaurants + 미식 방문형,2016,2017,2018,3724,527,3197,14.15
4,Restaurants + 미식 방문형,2017,2018,2019,4157,670,3487,16.12


In [62]:
# 5. 두 범위 결합 비교
rolling_scope_comparison_df = pd.concat(
    [
        restaurant_rolling_summary_df,
        restaurant_culinary_rolling_summary_df
    ],
    ignore_index=True
)

rolling_scope_comparison_df

,scope,selection_year,observation_year,target_year,candidate_users,churn_users,retained_users,churn_rate_pct
0,Restaurants,2013,2014,2015,2234,337,1897,15.09
1,Restaurants,2014,2015,2016,2762,444,2318,16.08
2,Restaurants,2015,2016,2017,3320,517,2803,15.57
3,Restaurants,2016,2017,2018,3537,510,3027,14.42
4,Restaurants,2017,2018,2019,3908,606,3302,15.51
5,Restaurants + 미식 방문형,2013,2014,2015,2324,363,1961,15.62
6,Restaurants + 미식 방문형,2014,2015,2016,2917,461,2456,15.80
7,Restaurants + 미식 방문형,2015,2016,2017,3513,557,2956,15.86
8,Restaurants + 미식 방문형,2016,2017,2018,3724,527,3197,14.15
9,Restaurants + 미식 방문형,2017,2018,2019,4157,670,3487,16.12


In [63]:
# 6. 전체 표본 증가량
restaurant_rolling_count = len(
    restaurant_rolling_df
)

culinary_rolling_count = len(
    restaurant_culinary_rolling_df
)

rolling_increase_count = (
    culinary_rolling_count
    - restaurant_rolling_count
)

rolling_increase_rate_pct = (
    rolling_increase_count
    / restaurant_rolling_count
    * 100
)

rolling_total_comparison_df = pd.DataFrame(
    {
        "scope": [
            "Restaurants",
            "Restaurants + 미식 방문형"
        ],
        "user_year_samples": [
            restaurant_rolling_count,
            culinary_rolling_count
        ],
        "unique_users": [
            restaurant_rolling_df[
                "user_id"
            ].nunique(),
            restaurant_culinary_rolling_df[
                "user_id"
            ].nunique()
        ],
        "churn_samples": [
            int(
                restaurant_rolling_df[
                    "churn"
                ].sum()
            ),
            int(
                restaurant_culinary_rolling_df[
                    "churn"
                ].sum()
            )
        ]
    }
)

rolling_total_comparison_df[
    "churn_rate_pct"
] = (
    rolling_total_comparison_df[
        "churn_samples"
    ]
    / rolling_total_comparison_df[
        "user_year_samples"
    ]
    * 100
).round(2)

rolling_total_comparison_df

,scope,user_year_samples,unique_users,churn_samples,churn_rate_pct
0,Restaurants,15761,9951,2414,15.32
1,Restaurants + 미식 방문형,16635,10483,2578,15.50


In [64]:
print(
    "Restaurants 롤링 표본:",
    f"{restaurant_rolling_count:,}"
)

print(
    "미식 확장 롤링 표본:",
    f"{culinary_rolling_count:,}"
)

print(
    "표본 증가:",
    f"{rolling_increase_count:,}"
)

print(
    "표본 증가율:",
    f"{rolling_increase_rate_pct:.2f}%"
)

Restaurants 롤링 표본: 15,761
미식 확장 롤링 표본: 16,635
표본 증가: 874
표본 증가율: 5.55%


In [65]:
# 먼저 2005~2017 전체를 계산하자
all_pre_covid_selection_years = list(
    range(
        2005,
        2018
    )
)

(
    culinary_all_pre_covid_rolling_df,
    culinary_all_pre_covid_summary_df
) = build_rolling_cohorts(
    annual_activity_df=
        restaurant_culinary_annual_df,
    second_half_activity_df=
        restaurant_culinary_second_half_df,
    selection_years=
        all_pre_covid_selection_years,
    scope_name=
        "Restaurants + 미식 방문형"
)

culinary_all_pre_covid_summary_df

,scope,selection_year,observation_year,target_year,candidate_users,churn_users,retained_users,churn_rate_pct
0,Restaurants + 미식 방문형,2005,2006,2007,2,0,2,0.00
1,Restaurants + 미식 방문형,2006,2007,2008,22,5,17,22.73
2,Restaurants + 미식 방문형,2007,2008,2009,86,17,69,19.77
3,Restaurants + 미식 방문형,2008,2009,2010,291,37,254,12.71
4,Restaurants + 미식 방문형,2009,2010,2011,550,78,472,14.18
5,Restaurants + 미식 방문형,2010,2011,2012,996,180,816,18.07
6,Restaurants + 미식 방문형,2011,2012,2013,1586,275,1311,17.34
7,Restaurants + 미식 방문형,2012,2013,2014,1834,275,1559,14.99
8,Restaurants + 미식 방문형,2013,2014,2015,2324,363,1961,15.62
9,Restaurants + 미식 방문형,2014,2015,2016,2917,461,2456,15.80


In [67]:
# 연도별 품질 조건 확인
MINIMUM_COHORT_USERS = 500
MINIMUM_CHURN_USERS = 50

culinary_all_pre_covid_summary_df[
    "enough_users"
] = (
    culinary_all_pre_covid_summary_df[
        "candidate_users"
    ]
    >= MINIMUM_COHORT_USERS
)

culinary_all_pre_covid_summary_df[
    "enough_churn_users"
] = (
    culinary_all_pre_covid_summary_df[
        "churn_users"
    ]
    >= MINIMUM_CHURN_USERS
)

culinary_all_pre_covid_summary_df[
    "usable_for_training"
] = (
    culinary_all_pre_covid_summary_df[
        "enough_users"
    ]
    & culinary_all_pre_covid_summary_df[
        "enough_churn_users"
    ]
)

culinary_all_pre_covid_summary_df

,scope,selection_year,observation_year,target_year,candidate_users,churn_users,retained_users,churn_rate_pct,enough_users,enough_churn_users,usable_for_training
0,Restaurants + 미식 방문형,2005,2006,2007,2,0,2,0.00,False,False,False
1,Restaurants + 미식 방문형,2006,2007,2008,22,5,17,22.73,False,False,False
2,Restaurants + 미식 방문형,2007,2008,2009,86,17,69,19.77,False,False,False
3,Restaurants + 미식 방문형,2008,2009,2010,291,37,254,12.71,False,False,False
4,Restaurants + 미식 방문형,2009,2010,2011,550,78,472,14.18,True,True,True
5,Restaurants + 미식 방문형,2010,2011,2012,996,180,816,18.07,True,True,True
6,Restaurants + 미식 방문형,2011,2012,2013,1586,275,1311,17.34,True,True,True
7,Restaurants + 미식 방문형,2012,2013,2014,1834,275,1559,14.99,True,True,True
8,Restaurants + 미식 방문형,2013,2014,2015,2324,363,1961,15.62,True,True,True
9,Restaurants + 미식 방문형,2014,2015,2016,2917,461,2456,15.80,True,True,True


In [68]:
usable_years = (
    culinary_all_pre_covid_summary_df
    .loc[
        culinary_all_pre_covid_summary_df[
            "usable_for_training"
        ],
        "selection_year"
    ]
    .tolist()
)

print(
    "학습 후보 선정연도:",
    usable_years
)

학습 후보 선정연도: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]


In [69]:
# 1. 최종 학습 후보 연도 확인

# 먼저 기존 결과에서 학습 가능한 연도를 가져온다
usable_years = (
    culinary_all_pre_covid_summary_df
    .loc[
        culinary_all_pre_covid_summary_df[
            "usable_for_training"
        ],
        "selection_year"
    ]
    .astype(int)
    .tolist()
)

# 2005년은 불완전한 연도라 기본 제외
usable_years = [
    year
    for year in usable_years
    if year >= 2006
]

print(
    "최종 학습 후보 연도:",
    usable_years
)

최종 학습 후보 연도: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]


In [70]:
# 2. 실험별 연도 정의
VALIDATION_YEAR = 2016
TEST_YEAR = 2017

# 실험 A: 품질 조건을 만족하는 모든 과거 연도
candidate_train_years = [
    year
    for year in usable_years
    if year <= 2015
]

# 실험 B: 최근 안정 구간
recent_train_years = [
    2013,
    2014,
    2015
]

print(
    "실험 A Train:",
    candidate_train_years
)

print(
    "실험 B Train:",
    recent_train_years
)

print(
    "Validation:",
    VALIDATION_YEAR
)

print(
    "Test:",
    TEST_YEAR
)

실험 A Train: [2009, 2010, 2011, 2012, 2013, 2014, 2015]
실험 B Train: [2013, 2014, 2015]
Validation: 2016
Test: 2017


In [71]:
assert VALIDATION_YEAR in usable_years
assert TEST_YEAR in usable_years

assert max(
    candidate_train_years
) < VALIDATION_YEAR

assert max(
    recent_train_years
) < VALIDATION_YEAR

assert VALIDATION_YEAR < TEST_YEAR

print("시간 순서 검증 통과")

시간 순서 검증 통과


In [72]:
# 3. 마스터 롤링 코호트 생성
master_rolling_cohort_df = (
    culinary_all_pre_covid_rolling_df
    [
        culinary_all_pre_covid_rolling_df[
            "selection_year"
        ].isin(
            usable_years
        )
    ]
    .copy()
    .reset_index(drop=True)
)

In [73]:
master_rolling_cohort_df[
    "sample_id"
] = (
    master_rolling_cohort_df[
        "user_id"
    ]
    + "_"
    + master_rolling_cohort_df[
        "selection_year"
    ].astype(str)
)

assert master_rolling_cohort_df[
    "sample_id"
].is_unique

assert master_rolling_cohort_df[
    [
        "user_id",
        "selection_year",
        "observation_year",
        "target_year",
        "churn"
    ]
].isna().sum().sum() == 0

print(
    "마스터 롤링 코호트:",
    master_rolling_cohort_df.shape
)

마스터 롤링 코호트: (21601, 11)


In [75]:
# 4. 실험 A 분할 표시
master_rolling_cohort_df[
    "candidate_range_split"
] = "excluded"

master_rolling_cohort_df.loc[
    master_rolling_cohort_df[
        "selection_year"
    ].isin(
        candidate_train_years
    ),
    "candidate_range_split"
] = "train"

master_rolling_cohort_df.loc[
    master_rolling_cohort_df[
        "selection_year"
    ]
    == VALIDATION_YEAR,
    "candidate_range_split"
] = "validation"

master_rolling_cohort_df.loc[
    master_rolling_cohort_df[
        "selection_year"
    ]
    == TEST_YEAR,
    "candidate_range_split"
] = "test"

In [76]:
# 5. 실험 B 분할 표시
master_rolling_cohort_df[
    "recent_range_split"
] = "excluded"

master_rolling_cohort_df.loc[
    master_rolling_cohort_df[
        "selection_year"
    ].isin(
        recent_train_years
    ),
    "recent_range_split"
] = "train"

master_rolling_cohort_df.loc[
    master_rolling_cohort_df[
        "selection_year"
    ]
    == VALIDATION_YEAR,
    "recent_range_split"
] = "validation"

master_rolling_cohort_df.loc[
    master_rolling_cohort_df[
        "selection_year"
    ]
    == TEST_YEAR,
    "recent_range_split"
] = "test"

In [77]:
# 6. 분할 결과 비교
def summarize_experiment_split(
    cohort_df,
    split_column,
    experiment_name
):
    summary_df = (
        cohort_df[
            cohort_df[
                split_column
            ]
            != "excluded"
        ]
        .groupby(
            split_column,
            as_index=False
        )
        .agg(
            samples=(
                "sample_id",
                "size"
            ),
            unique_users=(
                "user_id",
                "nunique"
            ),
            churn_users=(
                "churn",
                "sum"
            ),
            minimum_selection_year=(
                "selection_year",
                "min"
            ),
            maximum_selection_year=(
                "selection_year",
                "max"
            )
        )
        .rename(
            columns={
                split_column:
                    "split"
            }
        )
    )

    summary_df[
        "churn_rate_pct"
    ] = (
        summary_df[
            "churn_users"
        ]
        / summary_df[
            "samples"
        ]
        * 100
    ).round(2)

    summary_df.insert(
        0,
        "experiment",
        experiment_name
    )

    return summary_df

In [78]:
candidate_range_summary_df = (
    summarize_experiment_split(
        cohort_df=
            master_rolling_cohort_df,
        split_column=
            "candidate_range_split",
        experiment_name=
            "전체 학습 후보 연도"
    )
)

recent_range_summary_df = (
    summarize_experiment_split(
        cohort_df=
            master_rolling_cohort_df,
        split_column=
            "recent_range_split",
        experiment_name=
            "2013~2015 안정 구간"
    )
)

experiment_split_summary_df = (
    pd.concat(
        [
            candidate_range_summary_df,
            recent_range_summary_df
        ],
        ignore_index=True
    )
)

experiment_split_summary_df

,experiment,split,samples,unique_users,churn_users,minimum_selection_year,maximum_selection_year,churn_rate_pct
0,전체 학습 후보 연도,test,4157,4157,670,2017,2017,16.12
1,전체 학습 후보 연도,train,13720,8483,2189,2009,2015,15.95
2,전체 학습 후보 연도,validation,3724,3724,527,2016,2016,14.15
3,2013~2015 안정 구간,test,4157,4157,670,2017,2017,16.12
4,2013~2015 안정 구간,train,8754,6211,1381,2013,2015,15.78
5,2013~2015 안정 구간,validation,3724,3724,527,2016,2016,14.15


In [79]:
# 7. 시간 누수 검증
for split_column in [
    "candidate_range_split",
    "recent_range_split"
]:
    train_year_max = (
        master_rolling_cohort_df
        .loc[
            master_rolling_cohort_df[
                split_column
            ]
            == "train",
            "selection_year"
        ]
        .max()
    )

    validation_year_min = (
        master_rolling_cohort_df
        .loc[
            master_rolling_cohort_df[
                split_column
            ]
            == "validation",
            "selection_year"
        ]
        .min()
    )

    test_year_min = (
        master_rolling_cohort_df
        .loc[
            master_rolling_cohort_df[
                split_column
            ]
            == "test",
            "selection_year"
        ]
        .min()
    )

    assert (
        train_year_max
        < validation_year_min
        < test_year_min
    )

print("두 실험의 시간 분할 검증 통과")

두 실험의 시간 분할 검증 통과


In [80]:
# 8. 저장
ROLLING_COHORT_DIR = (
    INTERIM_DIR
    / "rolling"
)

ROLLING_COHORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MASTER_ROLLING_COHORT_PATH = (
    ROLLING_COHORT_DIR
    / "culinary_rolling_cohort_master_v02.parquet"
)

master_rolling_cohort_df.to_parquet(
    MASTER_ROLLING_COHORT_PATH,
    index=False
)

experiment_split_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "rolling_cohort_experiment_split_summary_v02.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "마스터 코호트 저장:",
    MASTER_ROLLING_COHORT_PATH
)

마스터 코호트 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\rolling\culinary_rolling_cohort_master_v02.parquet


In [81]:
# 그다음 단계

# 코호트 분할이 끝나면 바로 모델 학습으로 가는 것은 아니야. 다음에는 각 사용자-연도 표본에 맞춰 피처를 다시 생성해야 해.